<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day9_10_c_(260420)_%EA%B8%88%EC%9C%B5%EC%9D%B4%EC%B2%B4%EC%8B%A4%EC%8A%B5_%EB%8C%80%EC%8B%9C%EB%B3%B4%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile project-root/node-server/server.js

const express = require('express');
const cors = require('cors');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3001;

app.use(cors());
app.use(express.json());

app.use(express.static(path.join(__dirname, 'static')));

const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  waitForConnections: true,
  connectionLimit: 10,
  queueLimit: 0,
  dateStrings: true
});

const ACCOUNTS_GET_DELAY = 4000;
const TRANSFERS_GET_DELAY = 4000;
const ACCOUNTS_POST_DELAY = 2000;
const TRANSFERS_POST_DELAY = 5000;
const STALE_TIME = 10000;

function delay(ms) {
  return new Promise(function (resolve) {
    setTimeout(resolve, ms);
  });
}

function serverLog(message, payload) {
  const time = new Date().toLocaleTimeString('ko-KR');
  if (payload !== undefined) {
    console.log(`[SERVER ${time}] ${message}`, payload);
  } else {
    console.log(`[SERVER ${time}] ${message}`);
  }
}

app.get('/api/runtime-config', function (req, res) {
  res.json({
    staleTime: STALE_TIME,
    getDelay: {
      accounts: ACCOUNTS_GET_DELAY,
      transfers: TRANSFERS_GET_DELAY
    },
    postDelay: {
      accounts: ACCOUNTS_POST_DELAY,
      transfers: TRANSFERS_POST_DELAY
    }
  });
});

app.get('/api/accounts', async function (req, res) {
  try {
    serverLog('GET /api/accounts 요청 수신');
    await delay(ACCOUNTS_GET_DELAY);

    const [rows] = await pool.query(`
      SELECT
        id,
        owner_name,
        account_number,
        balance,
        created_at
      FROM accounts
      ORDER BY id ASC
    `);

    const viewedAt = new Date().toISOString();

    serverLog('GET /api/accounts 응답 전송', {
      count: rows.length,
      viewedAt
    });

    res.json({
      source: 'server',
      fetchedAt: viewedAt,
      items: rows.map(function (row) {
        return {
          ...row,
          viewed_at: viewedAt
        };
      })
    });
  } catch (error) {
    serverLog('GET /api/accounts 실패', error.message);
    res.status(500).json({
      message: '계좌 조회 실패'
    });
  }
});

app.get('/api/transfers', async function (req, res) {
  try {
    serverLog('GET /api/transfers 요청 수신');
    await delay(TRANSFERS_GET_DELAY);

    const [rows] = await pool.query(`
      SELECT
        t.id,
        t.sender_account_id,
        t.receiver_account_id,
        a1.owner_name AS sender_name,
        a1.account_number AS sender_account_number,
        a2.owner_name AS receiver_name,
        a2.account_number AS receiver_account_number,
        t.amount,
        t.transfer_type,
        t.status,
        t.created_at
      FROM transfers t
      JOIN accounts a1 ON t.sender_account_id = a1.id
      JOIN accounts a2 ON t.receiver_account_id = a2.id
      ORDER BY t.id DESC
    `);

    const viewedAt = new Date().toISOString();

    serverLog('GET /api/transfers 응답 전송', {
      count: rows.length,
      viewedAt
    });

    res.json({
      source: 'server',
      fetchedAt: viewedAt,
      items: rows.map(function (row) {
        return {
          ...row,
          viewed_at: viewedAt
        };
      })
    });
  } catch (error) {
    serverLog('GET /api/transfers 실패', error.message);
    res.status(500).json({
      message: '거래내역 조회 실패'
    });
  }
});

app.post('/api/accounts', async function (req, res) {
  try {
    serverLog('POST /api/accounts 요청 수신', req.body);

    const { owner_name, account_number, balance } = req.body;

    if (!owner_name || !account_number) {
      return res.status(400).json({
        message: '예금주명과 계좌번호는 필수입니다.'
      });
    }

    await delay(ACCOUNTS_POST_DELAY);

    await pool.query(
      `
        INSERT INTO accounts (
          owner_name,
          account_number,
          balance
        )
        VALUES (?, ?, ?)
      `,
      [owner_name, account_number, Number(balance) || 0]
    );

    serverLog('POST /api/accounts 등록 완료');

    res.json({
      message: '계좌 등록 완료'
    });
  } catch (error) {
    serverLog('POST /api/accounts 실패', error.message);
    res.status(500).json({
      message: '계좌 등록 실패'
    });
  }
});

app.post('/api/transfers', async function (req, res) {
  const connection = await pool.getConnection();

  try {
    serverLog('POST /api/transfers 요청 수신', req.body);

    const {
      sender_account_id,
      receiver_account_id,
      amount,
      transfer_type
    } = req.body;

    if (!sender_account_id || !receiver_account_id || !amount) {
      return res.status(400).json({
        message: '필수값이 누락되었습니다.'
      });
    }

    if (Number(sender_account_id) === Number(receiver_account_id)) {
      return res.status(400).json({
        message: '출금 계좌와 입금 계좌는 같을 수 없습니다.'
      });
    }

    await delay(TRANSFERS_POST_DELAY);

    await connection.beginTransaction();
    serverLog('트랜잭션 시작');

    const [senderRows] = await connection.query(
      'SELECT * FROM accounts WHERE id = ? FOR UPDATE',
      [sender_account_id]
    );

    const [receiverRows] = await connection.query(
      'SELECT * FROM accounts WHERE id = ? FOR UPDATE',
      [receiver_account_id]
    );

    if (senderRows.length === 0 || receiverRows.length === 0) {
      throw new Error('계좌를 찾을 수 없습니다.');
    }

    const sender = senderRows[0];
    const transferAmount = Number(amount);

    if (sender.balance < transferAmount) {
      throw new Error('잔액이 부족합니다.');
    }

    await connection.query(
      'UPDATE accounts SET balance = balance - ? WHERE id = ?',
      [transferAmount, sender_account_id]
    );

    await connection.query(
      'UPDATE accounts SET balance = balance + ? WHERE id = ?',
      [transferAmount, receiver_account_id]
    );

    await connection.query(
      `
        INSERT INTO transfers (
          sender_account_id,
          receiver_account_id,
          amount,
          transfer_type,
          status
        )
        VALUES (?, ?, ?, ?, '완료')
      `,
      [
        sender_account_id,
        receiver_account_id,
        transferAmount,
        transfer_type || '일반이체'
      ]
    );

    await connection.commit();
    serverLog('트랜잭션 커밋 완료');

    res.json({
      message: '이체 완료',
      completedAt: new Date().toISOString()
    });
  } catch (error) {
    await connection.rollback();
    serverLog('트랜잭션 롤백', error.message);

    res.status(500).json({
      message: error.message || '이체 실패'
    });
  } finally {
    connection.release();
  }
});

app.get('/', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  serverLog(`server running on port ${PORT}`);
});

Writing project-root/node-server/server.js


In [2]:
%%writefile project-root/node-server/reactQuery.sql

CREATE DATABASE IF NOT EXISTS testdb CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;
USE testdb;

DROP TABLE IF EXISTS transfers;
DROP TABLE IF EXISTS accounts;

CREATE TABLE accounts (
  id INT AUTO_INCREMENT PRIMARY KEY,
  owner_name VARCHAR(100) NOT NULL,
  account_number VARCHAR(30) NOT NULL UNIQUE,
  balance INT NOT NULL DEFAULT 0,
  created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE transfers (
  id INT AUTO_INCREMENT PRIMARY KEY,
  sender_account_id INT NOT NULL,
  receiver_account_id INT NOT NULL,
  amount INT NOT NULL,
  transfer_type VARCHAR(30) NOT NULL DEFAULT '일반이체',
  status VARCHAR(30) NOT NULL DEFAULT '완료',
  created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (sender_account_id) REFERENCES accounts(id),
  FOREIGN KEY (receiver_account_id) REFERENCES accounts(id)
);

INSERT INTO accounts (owner_name, account_number, balance) VALUES
('김민준', '100-200-300001', 500000),
('이서연', '100-200-300002', 700000),
('박지훈', '100-200-300003', 300000);

Writing project-root/node-server/reactQuery.sql


In [3]:
%%writefile project-root/react/index.html

<!doctype html>
<html lang="ko">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>React Query Finance Lab</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.jsx"></script>
  </body>
</html>

Overwriting project-root/react/index.html


In [4]:
%%writefile project-root/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { BrowserRouter } from 'react-router-dom';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './styles/app.css';

const queryClient = new QueryClient({
  defaultOptions: {
    queries: {
      retry: false,
      refetchOnWindowFocus: false,
      gcTime: 60000
    }
  }
});

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <BrowserRouter>
        <App />
      </BrowserRouter>
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting project-root/react/src/main.jsx


In [5]:
%%writefile project-root/react/src/App.jsx

import { Routes, Route, Link } from 'react-router-dom';
import { useEffect, useState } from 'react';
import { useMutation, useQuery, useQueryClient } from '@tanstack/react-query';
import {
  fetchAccounts,
  fetchTransfers,
  createAccount,
  createTransfer
} from './api/transfersApi';
import TransferForm from './components/TransferForm';
import TransferList from './components/TransferList';
import TransferSummary from './components/TransferSummary';

function DashboardPage() {
  const queryClient = useQueryClient();

  const [runtimeConfig, setRuntimeConfig] = useState({
    staleTime: 10000,
    getDelay: {
      accounts: 0,
      transfers: 0
    },
    postDelay: {
      accounts: 0,
      transfers: 0
    }
  });

  const STALE_TIME = runtimeConfig.staleTime;
  const [now, setNow] = useState(Date.now());
  const [logs, setLogs] = useState([]);

  function addLog(message, payload) {
    const time = new Date().toLocaleTimeString('ko-KR');
    const text = `[UI ${time}] ${message}`;

    if (payload !== undefined) {
      console.log(text, payload);
    } else {
      console.log(text);
    }

    setLogs(function (prev) {
      return [text, ...prev].slice(0, 30);
    });
  }

  useEffect(function () {
    addLog('대시보드 마운트 완료');

    const timer = setInterval(function () {
      setNow(Date.now());
    }, 1000);

    return function () {
      clearInterval(timer);
    };
  }, []);

  useEffect(function () {
    async function loadRuntimeConfig() {
      try {
        const response = await fetch('/api/runtime-config');
        const result = await response.json();
        setRuntimeConfig(result);
      } catch (error) {
        console.log('runtime config 조회 실패', error);
      }
    }

    loadRuntimeConfig();
  }, []);

  const accountsQuery = useQuery({
    queryKey: ['accounts'],
    queryFn: fetchAccounts,
    staleTime: STALE_TIME
  });

  const transfersQuery = useQuery({
    queryKey: ['transfers'],
    queryFn: fetchTransfers,
    staleTime: STALE_TIME
  });

  const accountsState = queryClient.getQueryState(['accounts']);
  const transfersState = queryClient.getQueryState(['transfers']);

  const accountsAge = accountsState?.dataUpdatedAt
    ? now - accountsState.dataUpdatedAt
    : 0;

  const transfersAge = transfersState?.dataUpdatedAt
    ? now - transfersState.dataUpdatedAt
    : 0;

  const accountsIsStale = accountsState?.dataUpdatedAt
    ? accountsAge >= STALE_TIME
    : false;

  const transfersIsStale = transfersState?.dataUpdatedAt
    ? transfersAge >= STALE_TIME
    : false;

  useEffect(function () {
    if (accountsQuery.isLoading) {
      addLog('accounts 최초 조회 진행 중');
    }
  }, [accountsQuery.isLoading]);

  useEffect(function () {
    if (transfersQuery.isLoading) {
      addLog('transfers 최초 조회 진행 중');
    }
  }, [transfersQuery.isLoading]);

  useEffect(function () {
    if (accountsQuery.isError) {
      addLog(`accounts 조회 실패 → ${accountsQuery.error.message}`);
    }
  }, [accountsQuery.isError, accountsQuery.error]);

  useEffect(function () {
    if (transfersQuery.isError) {
      addLog(`transfers 조회 실패 → ${transfersQuery.error.message}`);
    }
  }, [transfersQuery.isError, transfersQuery.error]);

  useEffect(function () {
    if (
      accountsState?.status === 'success' &&
      accountsIsStale &&
      !accountsQuery.isFetching
    ) {
      console.log('[RQ] accounts stale 감지 -> refetch 실행');
      addLog('accounts 캐시 만료 → 서버 재조회 시작');
      accountsQuery.refetch();
    }
  }, [
    accountsIsStale,
    accountsQuery,
    accountsQuery.isFetching,
    accountsState?.status
  ]);

  useEffect(function () {
    if (
      transfersState?.status === 'success' &&
      transfersIsStale &&
      !transfersQuery.isFetching
    ) {
      console.log('[RQ] transfers stale 감지 -> refetch 실행');
      addLog('transfers 캐시 만료 → 서버 재조회 시작');
      transfersQuery.refetch();
    }
  }, [
    transfersIsStale,
    transfersQuery,
    transfersQuery.isFetching,
    transfersState?.status
  ]);

  useEffect(function () {
    if (accountsQuery.isSuccess) {
      addLog('accounts 서버 응답 도착 → 캐시 저장 완료', {
        dataUpdatedAt: accountsQuery.dataUpdatedAt
      });
    }
  }, [accountsQuery.dataUpdatedAt, accountsQuery.isSuccess]);

  useEffect(function () {
    if (transfersQuery.isSuccess) {
      addLog('transfers 서버 응답 도착 → 캐시 저장 완료', {
        dataUpdatedAt: transfersQuery.dataUpdatedAt
      });
    }
  }, [transfersQuery.dataUpdatedAt, transfersQuery.isSuccess]);

  const createAccountMutation = useMutation({
    mutationFn: createAccount,
    onMutate: function (variables) {
      addLog('계좌 등록 요청 시작', variables);
    },
    onSuccess: function (data) {
      addLog('계좌 등록 성공 → accounts 캐시 무효화', data);
      console.log('[RQ] invalidateQueries -> accounts');
      queryClient.invalidateQueries({
        queryKey: ['accounts']
      });
    },
    onError: function (error) {
      addLog(`계좌 등록 실패 → ${error.message}`);
    }
  });

  const transferMutation = useMutation({
    mutationFn: createTransfer,
    onMutate: async function (newTransfer) {
      console.log('[RQ] onMutate 시작', newTransfer);
      addLog('이체 버튼 클릭 → Optimistic Update 시작', newTransfer);

      await queryClient.cancelQueries({
        queryKey: ['accounts']
      });

      console.log('[RQ] 기존 accounts 요청 취소 완료');

      const previousAccounts = queryClient.getQueryData(['accounts']);
      console.log('[RQ] 이전 캐시 백업', previousAccounts);

      queryClient.setQueryData(['accounts'], function (oldData) {
        if (!oldData || !oldData.items) {
          return oldData;
        }

        return {
          ...oldData,
          items: oldData.items.map(function (item) {
            if (item.id === Number(newTransfer.sender_account_id)) {
              return {
                ...item,
                balance: item.balance - Number(newTransfer.amount)
              };
            }

            if (item.id === Number(newTransfer.receiver_account_id)) {
              return {
                ...item,
                balance: item.balance + Number(newTransfer.amount)
              };
            }

            return item;
          })
        };
      });

      console.log('[RQ] Optimistic Update 적용 완료');
      addLog('화면 잔액 선반영 완료 → 서버 요청 전송');

      return {
        previousAccounts
      };
    },
    onError: function (error, variables, context) {
      console.log('[RQ] onError -> rollback 수행', error.message);
      queryClient.setQueryData(['accounts'], context.previousAccounts);
      addLog(`이체 실패 → 롤백 수행 → ${error.message}`);
    },
    onSuccess: function (data) {
      console.log('[RQ] onSuccess -> invalidateQueries accounts, transfers');
      addLog('이체 성공 → accounts, transfers 캐시 무효화', data);

      queryClient.invalidateQueries({
        queryKey: ['accounts']
      });

      queryClient.invalidateQueries({
        queryKey: ['transfers']
      });
    }
  });

  const accounts = accountsQuery.data?.items || [];
  const transfers = transfersQuery.data?.items || [];

  return (
    <div className="page-shell">
      <header className="hero">
        <div>
          <div className="hero-subtitle">React Query Finance Training</div>
          <h1>금융 이체 실습 대시보드</h1>
          <p>
            캐시 조회, stale 감지, 서버 재조회, 낙관적 업데이트, 캐시 무효화를
            시간 기준으로 관찰하는 실습 화면
          </p>
        </div>
        <div className="hero-side">
          <div className="hero-stat">
            <span>staleTime</span>
            <strong>{Math.ceil(runtimeConfig.staleTime / 1000)}초</strong>
          </div>
          <div className="hero-stat">
            <span>GET 지연</span>
            <strong>
              {Math.ceil(
                Math.max(
                  runtimeConfig.getDelay.accounts,
                  runtimeConfig.getDelay.transfers
                ) / 1000
              )}초
            </strong>
          </div>
          <div className="hero-stat">
            <span>POST 지연</span>
            <strong>
              {Math.ceil(
                Math.max(
                  runtimeConfig.postDelay.accounts,
                  runtimeConfig.postDelay.transfers
                ) / 1000
              )}초
            </strong>
          </div>
        </div>
      </header>

      <TransferSummary
        now={now}
        staleTime={STALE_TIME}
        accountsState={accountsState}
        transfersState={transfersState}
        accountsQuery={accountsQuery}
        transfersQuery={transfersQuery}
      />

      <TransferForm
        accounts={accounts}
        createAccountMutation={createAccountMutation}
        transferMutation={transferMutation}
      />

      <TransferList
        accounts={accounts}
        transfers={transfers}
        logs={logs}
      />
    </div>
  );
}

function AboutPage() {
  return (
    <div className="page-shell">
      <section className="panel">
        <div className="panel-header">
          <div>
            <h2>React Router 동작 확인</h2>
            <p>직접 /about 경로로 접근해도 서버가 index.html을 반환해야 한다</p>
          </div>
          <span className="badge badge-blue">Router</span>
        </div>

        <div className="guide-box">
          <p>이 페이지는 서버 fallback 검증용이다.</p>
          <p>브라우저 주소창에 /about 을 직접 입력해도 404가 아니라 정상 화면이 열려야 한다.</p>
          <p>그 이유는 node-server/server.js에 React Router fallback 코드가 있기 때문이다.</p>
        </div>
      </section>
    </div>
  );
}

function App() {
  return (
    <div className="app-layout">
      <aside className="sidebar">
        <div className="brand">FINANCE LAB</div>
        <nav className="menu">
          <Link to="/">대시보드</Link>
          <Link to="/about">라우터 확인</Link>
        </nav>
      </aside>

      <main className="main-area">
        <Routes>
          <Route path="/" element={<DashboardPage />} />
          <Route path="/about" element={<AboutPage />} />
        </Routes>
      </main>
    </div>
  );
}

export default App;

Overwriting project-root/react/src/App.jsx


In [6]:
%%writefile project-root/react/src/api/transfersApi.js

function logApi(message, payload) {
  const time = new Date().toLocaleTimeString('ko-KR');

  if (payload !== undefined) {
    console.log(`[API ${time}] ${message}`, payload);
  } else {
    console.log(`[API ${time}] ${message}`);
  }
}

export async function fetchAccounts() {
  logApi('GET /api/accounts 요청 시작');

  const response = await fetch('/api/accounts');

  logApi('GET /api/accounts 응답 도착', {
    status: response.status
  });

  if (!response.ok) {
    throw new Error('계좌 조회 실패');
  }

  const result = await response.json();
  logApi('GET /api/accounts JSON 파싱 완료', result);

  return result;
}

export async function fetchTransfers() {
  logApi('GET /api/transfers 요청 시작');

  const response = await fetch('/api/transfers');

  logApi('GET /api/transfers 응답 도착', {
    status: response.status
  });

  if (!response.ok) {
    throw new Error('거래내역 조회 실패');
  }

  const result = await response.json();
  logApi('GET /api/transfers JSON 파싱 완료', result);

  return result;
}

export async function createAccount(form) {
  logApi('POST /api/accounts 요청 시작', form);

  const response = await fetch('/api/accounts', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(form)
  });

  const result = await response.json();

  logApi('POST /api/accounts 응답 도착', {
    status: response.status,
    result
  });

  if (!response.ok) {
    throw new Error(result.message || '계좌 등록 실패');
  }

  return result;
}

export async function createTransfer(form) {
  logApi('POST /api/transfers 요청 시작', form);

  const response = await fetch('/api/transfers', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(form)
  });

  const result = await response.json();

  logApi('POST /api/transfers 응답 도착', {
    status: response.status,
    result
  });

  if (!response.ok) {
    throw new Error(result.message || '이체 실패');
  }

  return result;
}

Writing project-root/react/src/api/transfersApi.js


In [7]:
%%writefile project-root/react/src/components/TransferForm.jsx

import { useMemo, useState } from 'react';

function formatNumber(value) {
  return Number(value).toLocaleString('ko-KR');
}

function TransferForm(props) {
  const {
    accounts,
    createAccountMutation,
    transferMutation
  } = props;

  const [accountForm, setAccountForm] = useState({
    owner_name: '',
    account_number: '',
    balance: ''
  });

  const [transferForm, setTransferForm] = useState({
    sender_account_id: '',
    receiver_account_id: '',
    amount: '',
    transfer_type: '일반이체'
  });

  const selectedSender = useMemo(function () {
    return accounts.find(function (item) {
      return item.id === Number(transferForm.sender_account_id);
    });
  }, [accounts, transferForm.sender_account_id]);

  return (
    <div className="content-grid">
      <section className="panel form-panel">
        <div className="panel-header">
          <div>
            <h2>계좌 등록</h2>
            <p>신규 계좌 생성 후 accounts 캐시 무효화</p>
          </div>
          <span className="badge badge-green">Create</span>
        </div>

        <div className="field-group">
          <label>예금주명</label>
          <input
            type="text"
            placeholder="예: 한지민"
            value={accountForm.owner_name}
            onChange={function (e) {
              setAccountForm({
                ...accountForm,
                owner_name: e.target.value
              });
            }}
          />
        </div>

        <div className="field-group">
          <label>계좌번호</label>
          <input
            type="text"
            placeholder="예: 100-200-399999"
            value={accountForm.account_number}
            onChange={function (e) {
              setAccountForm({
                ...accountForm,
                account_number: e.target.value
              });
            }}
          />
        </div>

        <div className="field-group">
          <label>초기 잔액</label>
          <input
            type="number"
            placeholder="예: 1000000"
            value={accountForm.balance}
            onChange={function (e) {
              setAccountForm({
                ...accountForm,
                balance: e.target.value
              });
            }}
          />
        </div>

        <button
          className="primary-button"
          onClick={function () {
            createAccountMutation.mutate(accountForm, {
              onSuccess: function () {
                setAccountForm({
                  owner_name: '',
                  account_number: '',
                  balance: ''
                });
              }
            });
          }}
          disabled={createAccountMutation.isPending}
        >
          {createAccountMutation.isPending ? '등록 처리 중...' : '계좌 등록'}
        </button>
      </section>

      <section className="panel form-panel">
        <div className="panel-header">
          <div>
            <h2>이체 실행</h2>
            <p>Optimistic Update와 Query Invalidation 관찰</p>
          </div>
          <span className="badge badge-red">Transfer</span>
        </div>

        <div className="field-group">
          <label>출금 계좌</label>
          <select
            value={transferForm.sender_account_id}
            onChange={function (e) {
              setTransferForm({
                ...transferForm,
                sender_account_id: e.target.value
              });
            }}
          >
            <option value="">출금 계좌 선택</option>
            {accounts.map(function (account) {
              return (
                <option key={account.id} value={account.id}>
                  {account.owner_name} / {account.account_number}
                </option>
              );
            })}
          </select>
        </div>

        <div className="field-group">
          <label>입금 계좌</label>
          <select
            value={transferForm.receiver_account_id}
            onChange={function (e) {
              setTransferForm({
                ...transferForm,
                receiver_account_id: e.target.value
              });
            }}
          >
            <option value="">입금 계좌 선택</option>
            {accounts.map(function (account) {
              return (
                <option key={account.id} value={account.id}>
                  {account.owner_name} / {account.account_number}
                </option>
              );
            })}
          </select>
        </div>

        <div className="field-group">
          <label>이체 금액</label>
          <input
            type="number"
            placeholder="예: 50000"
            value={transferForm.amount}
            onChange={function (e) {
              setTransferForm({
                ...transferForm,
                amount: e.target.value
              });
            }}
          />
        </div>

        <div className="field-group">
          <label>이체 유형</label>
          <select
            value={transferForm.transfer_type}
            onChange={function (e) {
              setTransferForm({
                ...transferForm,
                transfer_type: e.target.value
              });
            }}
          >
            <option value="일반이체">일반이체</option>
            <option value="급여이체">급여이체</option>
            <option value="자동이체">자동이체</option>
          </select>
        </div>

        <div className="preview-card">
          <div className="preview-title">출금 계좌 현재 잔액</div>
          <div className="preview-value">
            {selectedSender ? `${formatNumber(selectedSender.balance)}원` : '-'}
          </div>
        </div>

        <button
          className="danger-button"
          onClick={function () {
            transferMutation.mutate(transferForm, {
              onSuccess: function () {
                setTransferForm({
                  sender_account_id: '',
                  receiver_account_id: '',
                  amount: '',
                  transfer_type: '일반이체'
                });
              }
            });
          }}
          disabled={transferMutation.isPending}
        >
          {transferMutation.isPending ? '이체 처리 중...' : '이체 실행'}
        </button>
      </section>
    </div>
  );
}

export default TransferForm;

Writing project-root/react/src/components/TransferForm.jsx


In [8]:
%%writefile project-root/react/src/components/TransferList.jsx

function formatNumber(value) {
  return Number(value).toLocaleString('ko-KR');
}

function formatDateTime(value) {
  if (!value) {
    return '-';
  }

  return new Date(value).toLocaleString('ko-KR');
}

function TransferList(props) {
  const {
    accounts,
    transfers,
    logs
  } = props;

  return (
    <div className="table-grid">
      <section className="panel">
        <div className="panel-header">
          <div>
            <h2>계좌 현황</h2>
            <p>현재 목록에 표시된 시간은 DB 생성시각이 아니라 이번 조회 응답 시각이다</p>
          </div>
          <span className="badge badge-gray">{accounts.length}건</span>
        </div>

        <div className="table-wrap">
          <table>
            <thead>
              <tr>
                <th>ID</th>
                <th>예금주</th>
                <th>계좌번호</th>
                <th>잔액</th>
                <th>조회 반영 시각</th>
              </tr>
            </thead>
            <tbody>
              {accounts.map(function (account) {
                return (
                  <tr key={account.id}>
                    <td>{account.id}</td>
                    <td>{account.owner_name}</td>
                    <td>{account.account_number}</td>
                    <td className="amount-cell">{formatNumber(account.balance)}원</td>
                    <td>{formatDateTime(account.viewed_at)}</td>
                  </tr>
                );
              })}
            </tbody>
          </table>
        </div>
      </section>

      <section className="panel">
        <div className="panel-header">
          <div>
            <h2>거래 내역</h2>
            <p>현재 목록에 표시된 시간은 DB 저장시각이 아니라 이번 조회 응답 시각이다</p>
          </div>
          <span className="badge badge-gray">{transfers.length}건</span>
        </div>

        <div className="table-wrap">
          <table>
            <thead>
              <tr>
                <th>ID</th>
                <th>출금</th>
                <th>입금</th>
                <th>금액</th>
                <th>유형</th>
                <th>상태</th>
                <th>조회 반영 시각</th>
              </tr>
            </thead>
            <tbody>
              {transfers.map(function (transfer) {
                return (
                  <tr key={transfer.id}>
                    <td>{transfer.id}</td>
                    <td>{transfer.sender_name}</td>
                    <td>{transfer.receiver_name}</td>
                    <td className="amount-cell">{formatNumber(transfer.amount)}원</td>
                    <td>{transfer.transfer_type}</td>
                    <td>{transfer.status}</td>
                    <td>{formatDateTime(transfer.viewed_at)}</td>
                  </tr>
                );
              })}
            </tbody>
          </table>
        </div>
      </section>

      <section className="panel">
        <div className="panel-header">
          <div>
            <h2>이벤트 로그</h2>
            <p>화면 로그와 console.log를 함께 비교</p>
          </div>
          <span className="badge badge-gray">{logs.length}건</span>
        </div>

        <div className="log-box">
          {logs.map(function (log, index) {
            return (
              <div key={index} className="log-line">
                {log}
              </div>
            );
          })}
        </div>
      </section>
    </div>
  );
}

export default TransferList;

Writing project-root/react/src/components/TransferList.jsx


In [9]:
%%writefile project-root/react/src/components/TransferSummary.jsx

function formatTime(value) {
  if (!value) {
    return '-';
  }

  return new Date(value).toLocaleTimeString('ko-KR');
}

function TransferSummary(props) {
  const {
    now,
    staleTime,
    accountsState,
    transfersState,
    accountsQuery,
    transfersQuery
  } = props;

  const accountsAge = accountsState?.dataUpdatedAt
    ? now - accountsState.dataUpdatedAt
    : 0;

  const transfersAge = transfersState?.dataUpdatedAt
    ? now - transfersState.dataUpdatedAt
    : 0;

  const accountsRemain = accountsState?.dataUpdatedAt
    ? Math.max(0, staleTime - accountsAge)
    : 0;

  const transfersRemain = transfersState?.dataUpdatedAt
    ? Math.max(0, staleTime - transfersAge)
    : 0;

  const accountsStatus = accountsAge >= staleTime ? 'stale' : 'fresh';
  const transfersStatus = transfersAge >= staleTime ? 'stale' : 'fresh';

  return (
    <section className="panel summary-panel">
      <div className="panel-header">
        <div>
          <h2>캐시 상태 모니터링</h2>
          <p>최초 목록 표시 이후 현재 목록을 유지한 채 서버 재검증 여부를 확인</p>
        </div>
        <span className="badge badge-blue">React Query</span>
      </div>

      <div className="summary-grid">
        <div className="summary-card">
          <div className="summary-label">현재 시간</div>
          <div className="summary-value">{formatTime(now)}</div>
          <div className="summary-meta">staleTime {Math.ceil(staleTime / 1000)}초</div>
        </div>

        <div className="summary-card">
          <div className="summary-label">accounts</div>
          <div className="summary-value">{formatTime(accountsState?.dataUpdatedAt)}</div>
          <div className="summary-meta">남은 유효시간 {Math.ceil(accountsRemain / 1000)}초</div>
          <div className="summary-meta">상태 {accountsStatus}</div>
          <div className="summary-meta">조회 중 {accountsQuery.isFetching ? '예' : '아니오'}</div>

          <div className="button-row">
            <button
              type="button"
              className="secondary-button"
              onClick={function () {
                accountsQuery.refetch();
              }}
              disabled={accountsQuery.isFetching}
            >
              accounts 목록조회
            </button>
          </div>
        </div>

        <div className="summary-card">
          <div className="summary-label">transfers</div>
          <div className="summary-value">{formatTime(transfersState?.dataUpdatedAt)}</div>
          <div className="summary-meta">남은 유효시간 {Math.ceil(transfersRemain / 1000)}초</div>
          <div className="summary-meta">상태 {transfersStatus}</div>
          <div className="summary-meta">조회 중 {transfersQuery.isFetching ? '예' : '아니오'}</div>

          <div className="button-row">
            <button
              type="button"
              className="secondary-button"
              onClick={function () {
                transfersQuery.refetch();
              }}
              disabled={transfersQuery.isFetching}
            >
              transfers 목록조회
            </button>
          </div>
        </div>
      </div>

      <div className="button-row top-gap">
        <button
          type="button"
          className="primary-outline-button"
          onClick={function () {
            accountsQuery.refetch();
            transfersQuery.refetch();
          }}
          disabled={accountsQuery.isFetching || transfersQuery.isFetching}
        >
          전체 목록조회
        </button>
      </div>
    </section>
  );
}

export default TransferSummary;

Writing project-root/react/src/components/TransferSummary.jsx


In [ ]:
%%writefile project-root/react/src/styles/app.css

* {
  box-sizing: border-box;
}

:root {
  --bg: #f3f6fb;
  --panel: #ffffff;
  --line: #d9e2ec;
  --text: #1f2937;
  --muted: #6b7280;
  --navy: #0f172a;
  --blue: #2563eb;
  --green: #059669;
  --red: #dc2626;
  --gray: #64748b;
  --shadow: 0 10px 30px rgba(15, 23, 42, 0.08);
}

body {
  margin: 0;
  background: var(--bg);
  color: var(--text);
  font-family: 'Segoe UI', Arial, sans-serif;
}

a {
  text-decoration: none;
  color: inherit;
}

.app-layout {
  display: grid;
  grid-template-columns: 240px 1fr;
  min-height: 100vh;
}

.sidebar {
  background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%);
  color: #ffffff;
  padding: 28px 20px;
}

.brand {
  font-size: 24px;
  font-weight: 800;
  letter-spacing: 1px;
  margin-bottom: 32px;
}

.menu {
  display: flex;
  flex-direction: column;
  gap: 10px;
}

.menu a {
  padding: 12px 14px;
  border-radius: 10px;
  color: #e2e8f0;
  background: rgba(255, 255, 255, 0.04);
}

.menu a:hover {
  background: rgba(255, 255, 255, 0.1);
}

.main-area {
  padding: 24px;
}

.page-shell {
  max-width: 1440px;
  margin: 0 auto;
}

.hero {
  display: flex;
  justify-content: space-between;
  gap: 24px;
  padding: 28px;
  border-radius: 18px;
  background: linear-gradient(135deg, #1d4ed8 0%, #0f172a 100%);
  color: #ffffff;
  box-shadow: var(--shadow);
  margin-bottom: 24px;
}

.hero h1 {
  margin: 8px 0 10px;
  font-size: 32px;
}

.hero p {
  margin: 0;
  color: #dbeafe;
  line-height: 1.6;
}

.hero-subtitle {
  font-size: 13px;
  letter-spacing: 1.5px;
  text-transform: uppercase;
  color: #bfdbfe;
}

.hero-side {
  display: grid;
  grid-template-columns: 1fr;
  gap: 12px;
  min-width: 180px;
}

.hero-stat {
  background: rgba(255, 255, 255, 0.08);
  border: 1px solid rgba(255, 255, 255, 0.14);
  border-radius: 14px;
  padding: 14px;
}

.hero-stat span {
  display: block;
  font-size: 13px;
  color: #cbd5e1;
  margin-bottom: 6px;
}

.hero-stat strong {
  font-size: 22px;
}

.panel {
  background: var(--panel);
  border: 1px solid var(--line);
  border-radius: 18px;
  padding: 22px;
  box-shadow: var(--shadow);
  margin-bottom: 20px;
}

.panel-header {
  display: flex;
  justify-content: space-between;
  gap: 16px;
  align-items: flex-start;
  margin-bottom: 18px;
}

.panel-header h2 {
  margin: 0 0 6px;
  font-size: 22px;
}

.panel-header p {
  margin: 0;
  color: var(--muted);
  line-height: 1.5;
}

.badge {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  min-width: 72px;
  height: 32px;
  padding: 0 12px;
  border-radius: 999px;
  font-size: 13px;
  font-weight: 700;
}

.badge-blue {
  background: #dbeafe;
  color: #1d4ed8;
}

.badge-green {
  background: #d1fae5;
  color: #047857;
}

.badge-red {
  background: #fee2e2;
  color: #b91c1c;
}

.badge-gray {
  background: #e2e8f0;
  color: #334155;
}

.summary-panel {
  margin-bottom: 24px;
}

.summary-grid {
  display: grid;
  grid-template-columns: 1fr 1fr 1fr;
  gap: 16px;
}

.summary-card {
  background: #f8fafc;
  border: 1px solid #e5edf5;
  border-radius: 16px;
  padding: 18px;
}

.summary-label {
  font-size: 13px;
  color: var(--muted);
  margin-bottom: 8px;
}

.summary-value {
  font-size: 24px;
  font-weight: 800;
  margin-bottom: 8px;
}

.summary-meta {
  font-size: 14px;
  color: #475569;
  margin-top: 5px;
}

.content-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 20px;
}

.form-panel {
  min-height: 100%;
}

.field-group {
  margin-bottom: 14px;
}

.field-group label {
  display: block;
  margin-bottom: 8px;
  font-size: 14px;
  font-weight: 700;
  color: #334155;
}

input,
select {
  width: 100%;
  height: 46px;
  padding: 0 14px;
  border: 1px solid #cbd5e1;
  border-radius: 12px;
  background: #ffffff;
  font-size: 14px;
  outline: none;
}

input:focus,
select:focus {
  border-color: #2563eb;
  box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.12);
}

.primary-button,
.danger-button {
  width: 100%;
  height: 48px;
  border: none;
  border-radius: 12px;
  color: #ffffff;
  font-size: 15px;
  font-weight: 700;
  cursor: pointer;
  margin-top: 8px;
}

.primary-button {
  background: linear-gradient(135deg, #059669 0%, #047857 100%);
}

.danger-button {
  background: linear-gradient(135deg, #dc2626 0%, #b91c1c 100%);
}

.primary-button:disabled,
.danger-button:disabled {
  background: #94a3b8;
  cursor: not-allowed;
}

.preview-card {
  margin: 12px 0 6px;
  padding: 16px;
  border-radius: 14px;
  background: linear-gradient(135deg, #eff6ff 0%, #dbeafe 100%);
  border: 1px solid #bfdbfe;
}

.preview-title {
  font-size: 13px;
  color: #1e3a8a;
  margin-bottom: 6px;
}

.preview-value {
  font-size: 24px;
  font-weight: 800;
  color: #1d4ed8;
}

.table-grid {
  display: grid;
  grid-template-columns: 1fr;
  gap: 20px;
}

.table-wrap {
  overflow-x: auto;
}

table {
  width: 100%;
  border-collapse: collapse;
}

th,
td {
  border-bottom: 1px solid #e5e7eb;
  padding: 14px 12px;
  text-align: left;
  font-size: 14px;
}

th {
  background: #f8fafc;
  color: #334155;
  font-weight: 800;
}

tr:hover td {
  background: #f8fbff;
}

.amount-cell {
  font-weight: 800;
  color: #0f172a;
}

.log-box {
  max-height: 320px;
  overflow-y: auto;
  padding: 14px;
  border-radius: 14px;
  background: #0f172a;
  color: #e2e8f0;
}

.log-line {
  font-size: 13px;
  line-height: 1.6;
  padding-bottom: 8px;
  margin-bottom: 8px;
  border-bottom: 1px solid rgba(255, 255, 255, 0.08);
}

.guide-box {
  border-radius: 14px;
  padding: 18px;
  background: #f8fafc;
  border: 1px solid #e2e8f0;
}

.guide-box p {
  margin: 0 0 10px;
  line-height: 1.7;
}

@media (max-width: 1200px) {
  .summary-grid {
    grid-template-columns: 1fr;
  }

  .content-grid {
    grid-template-columns: 1fr;
  }

  .hero {
    flex-direction: column;
  }
}

@media (max-width: 900px) {
  .app-layout {
    grid-template-columns: 1fr;
  }

  .sidebar {
    padding: 16px;
  }

  .menu {
    flex-direction: row;
    flex-wrap: wrap;
  }

  .main-area {
    padding: 16px;
  }

  .hero h1 {
    font-size: 26px;
  }
}